In [1]:
import os
import sys
import torch as t
import pandas as pd
import numpy as np

PROJECT_ROOT = r"C:\Mestrado\Graph_Pruning\AnyGraph"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
sys.argv = ["analysis"]

In [2]:
from node_classification.model import AnyGraph
from node_classification.params import args
import numpy as np
import warnings
import Utils.TimeLogger as logger

In [3]:
from Pruning_Methods.globalmagnitudepruner import GlobalMagnitudePruner

In [5]:
def load_model_from_checkpoint(checkpoint_path):
    #Configuração mínima que o model.py espera
    args.devices = ["cuda:0", "cuda:0"]

    #Carregando o modelo
    checkpoint = t.load(f"Models/{checkpoint_path}.mod", weights_only=False)
    model = checkpoint["model"]
    return model 

In [6]:
model1 = load_model_from_checkpoint("pretrain_link1")
print(type(model1))

<class 'model.AnyGraph'>


In [7]:
pruner = GlobalMagnitudePruner(model1)

In [8]:
pruner.prune(0.30)
pruner.sparsity_report()

Threshold = 0.005317


{'Total': 16777216,
 'Zero': 5033166,
 'Remaining': 11744050,
 'Sparsity (%)': 30.000007152557373}

In [9]:
pruner.expert_report()

,Expert,Weights,Zeros,Sparsity (%)
0,0,2097152,876902,41.813946
1,1,2097152,455132,21.702385
2,2,2097152,888585,42.371035
3,3,2097152,659056,31.426239
4,4,2097152,339115,16.170263
5,5,2097152,548595,26.159048
6,6,2097152,398099,18.982840
7,7,2097152,867682,41.374302


In [19]:
df=pruner.layer_report()
df.head(10)

,Expert,Layer,Weights,Zeros,Sparsity (%)
0,0,0,262144,55201,21.057510
1,0,1,262144,38750,14.781952
2,0,2,262144,35985,13.727188
3,0,3,262144,65345,24.927139
4,0,4,262144,121914,46.506500
5,0,5,262144,161940,61.775208
6,0,6,262144,188354,71.851349
7,0,7,262144,209413,79.884720
8,1,0,262144,38978,14.868927
9,1,1,262144,35930,13.706207


In [11]:
pruner.remaining_statistics()

{'Remaining': 11744050,
 'Mean': 0.0011399935465306044,
 'Std': 0.03000212460756302,
 'Median': 0.006070356350392103,
 'Min': -0.6176326870918274,
 'Max': 0.6792733073234558}

In [4]:
from node_classification.data_handler import MultiDataHandler
from node_classification.main import Exp   

In [5]:
datasets = {
    "node": [
        "cora",
        "arxiv",
        "pubmed",
        "home",
        "tech"
    ]
}

trn_datasets = datasets["node"]
tst_datasets = datasets["node"]

In [6]:
# Igual ao main.py
t.sparse.check_sparse_tensor_invariants.disable()

warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    message="divide by zero encountered in power"
)

os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu

if len(args.gpu.split(",")) == 2:
    args.devices = ["cuda:0", "cuda:1"]
elif len(args.gpu.split(",")) > 2:
    raise Exception("Devices should be less than 2")
else:
    args.devices = ["cuda:0", "cuda:0"]

logger.saveDefault = True

print(args)
print("Devices:", args.devices)

Namespace(lr=0.0001, batch=4096, tst_batch=256, epoch=100, save_path='tem', load_model=None, data='ml1m', tst_epoch=1, gpu='0', topk=20, epoch_max_step=-1, trn_mode='train-all', tst_mode='tst', eval_loss=True, ratio_fewshot_set=0.5, shot=5, tst_steps=-1, reg=1e-07, latdim=512, gnn_layer=3, fc_layer=8, gt_layer=2, head=4, anchor=256, act='relu', dataset_setting='training', assignment='top1', scale_layer=10, leaky=0.5, drop_rate=0.1, reca_range=0.2, selfloop=0, niter=2, expert_num=8, loss='ce', proj_method='both', nn='mlp', proj_trn_steps=100, attempt_cache=10000000, devices=['cuda:0', 'cuda:0'])
Devices: ['cuda:0', 'cuda:0']


In [7]:
handler = MultiDataHandler(
    trn_datasets,
    [tst_datasets]
)

2026-08-12 21:46:17.335444: Loading dataset arxiv
Dataset: arxiv, Node num: 169383, Edge num: 1456326
2026-08-12 21:46:22.246547: Loading dataset cora
Dataset: cora, Node num: 25191, Edge num: 121912
2026-08-12 21:46:22.709474: Loading dataset home
Dataset: home, Node num: 9795, Edge num: 149463
2026-08-12 21:46:23.008325: Loading dataset pubmed
Dataset: pubmed, Node num: 19720, Edge num: 124138
2026-08-12 21:46:23.311176: Loading dataset tech
Dataset: tech, Node num: 47431, Edge num: 2162611


Baseline

In [8]:
checkpoint="pretrain_link1"

exp = Exp(handler)

exp.prepare_model()

exp.load_model(checkpoint)

df = exp.evaluate()

print(df)

['arxiv', 'cora', 'home', 'pubmed', 'tech']
Test group 0 ['arxiv', 'cora', 'home', 'pubmed', 'tech']
Total params: 16.87552
Trainable params: 16.87552
Non-trainable params: 0.0
2026-08-10 17:03:01.387425: Loaded pretrain_link1
2026-08-10 17:03:22.852121: Epoch 100/100, arxiv Test: Acc_mean = 0.4998, Acc_std = 0.0244, F1_mean = 0.3032, F1_std = 0.0096, tstNum_mean = 48603.0000, tstNum_std = 0.0000      
2026-08-10 17:03:25.731794: Epoch 100/100, cora Test: Acc_mean = 0.6185, Acc_std = 0.0088, F1_mean = 0.5621, F1_std = 0.0063, tstNum_mean = 3414.0000, tstNum_std = 0.0000      
2026-08-10 17:03:27.321507: Epoch 100/100, home Test: Acc_mean = 0.6678, Acc_std = 0.0407, F1_mean = 0.4327, F1_std = 0.0744, tstNum_mean = 1958.0000, tstNum_std = 0.0000      
2026-08-10 17:03:29.460876: Epoch 100/100, pubmed Test: Acc_mean = 0.6921, Acc_std = 0.0699, F1_mean = 0.6435, F1_std = 0.0770, tstNum_mean = 3944.0000, tstNum_std = 0.0000      
2026-08-10 17:06:20.220657: Epoch 100/100, tech Test: Acc_mea

Pruning 90% link1

In [15]:
checkpoint="pretrain_link1"

exp = Exp(handler)

exp.prepare_model()

exp.load_model(checkpoint)

# Modelo pré-treinado
model_original = exp.model

# Cria uma cópia e poda essa cópia
pruner_mine = GlobalMagnitudePruner(model_original)

# Modelo novo, explicitamente podado
model_pruned = pruner_mine.prune(0.90)

# Passa explicitamente o modelo podado para o Exp
exp.model = model_pruned

# Confirma
print(pruner_mine.sparsity_report())

# Avalia o modelo PODADO
df = exp.evaluate()

print(df)

['arxiv', 'cora', 'home', 'pubmed', 'tech']
Test group 0 ['arxiv', 'cora', 'home', 'pubmed', 'tech']
Total params: 16.87552
Trainable params: 16.87552
Non-trainable params: 0.0
2026-08-12 16:08:24.181141: Loaded pretrain_link1
Global Pruning: 90.0%
Threshold: 0.04173007
{'Threshold': 0.04173006862401962, 'Requested Sparsity (%)': 90.0, 'Total': 16777216, 'Zero': 15099494, 'Remaining': 1677722, 'Sparsity (%)': 89.99999761581421}
2026-08-12 16:08:49.757560: Epoch 100/100, arxiv Test: Acc_mean = 0.4797, Acc_std = 0.1136, F1_mean = 0.3469, F1_std = 0.0646, tstNum_mean = 48603.0000, tstNum_std = 0.0000      
2026-08-12 16:08:53.538686: Epoch 100/100, cora Test: Acc_mean = 0.5677, Acc_std = 0.0085, F1_mean = 0.5157, F1_std = 0.0151, tstNum_mean = 3414.0000, tstNum_std = 0.0000      
2026-08-12 16:08:55.793997: Epoch 100/100, home Test: Acc_mean = 0.5860, Acc_std = 0.0564, F1_mean = 0.3572, F1_std = 0.0691, tstNum_mean = 1958.0000, tstNum_std = 0.0000      
2026-08-12 16:08:58.544963: Epoch 1

In [16]:
model_mine = exp.model

In [9]:
print(
    "Zeros no exp.model:"
)

total = 0
zeros = 0

for name, param in exp.model.named_parameters():

    if (
        "trainable_nn" in name
        and "linear.weight" in name
    ):

        total += param.numel()
        zeros += (param == 0).sum().item()

print("Total:", total)
print("Zeros:", zeros)
print(
    "Sparsity:",
    100 * zeros / total
)

Zeros no exp.model:
Total: 16777216
Zeros: 15099494
Sparsity: 89.99999761581421


Pruning 90% pytorch link1

In [8]:
import torch.nn.utils.prune as prune

In [9]:
exp = Exp(handler)

exp.prepare_model()

exp.load_model("pretrain_link1")


['arxiv', 'cora', 'home', 'pubmed', 'tech']
Test group 0 ['arxiv', 'cora', 'home', 'pubmed', 'tech']
Total params: 16.87552
Trainable params: 16.87552
Non-trainable params: 0.0
2026-08-12 15:53:05.020667: Loaded pretrain_link1


In [10]:
parameters_to_prune = []

for name, module in exp.model.named_modules():

    if isinstance(module, t.nn.Linear):

        if "trainable_nn" in name:

            parameters_to_prune.append(
                (module, "weight")
            )

print(len(parameters_to_prune))

64


In [11]:
total = 0
zeros = 0

for module, name in parameters_to_prune:

    weight = getattr(module, name)

    total += weight.numel()
    zeros += (weight == 0).sum().item()

print("Total:", total)
print("Zeros:", zeros)
print("Sparsity:", 100 * zeros / total)

Total: 16777216
Zeros: 0
Sparsity: 0.0


In [12]:
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.90
)

In [13]:
total = 0
zeros = 0

for module, name in parameters_to_prune:

    weight = getattr(module, name)

    total += weight.numel()
    zeros += (weight == 0).sum().item()

print("Total:", total)
print("Zeros:", zeros)
print("Sparsity:", 100 * zeros / total)

Total: 16777216
Zeros: 15099494
Sparsity: 89.99999761581421


In [17]:
exp = Exp(handler)

exp.prepare_model()

exp.load_model("pretrain_link1")

parameters_to_prune = []

for name, module in exp.model.named_modules():

    if (
        isinstance(module, t.nn.Linear)
        and "trainable_nn" in name
    ):
        parameters_to_prune.append(
            (module, "weight")
        )

prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.90
)

df = exp.evaluate()

print(df)

['arxiv', 'cora', 'home', 'pubmed', 'tech']
Test group 0 ['arxiv', 'cora', 'home', 'pubmed', 'tech']
Total params: 16.87552
Trainable params: 16.87552
Non-trainable params: 0.0
2026-08-12 16:12:52.179452: Loaded pretrain_link1
2026-08-12 16:13:17.008332: Epoch 100/100, arxiv Test: Acc_mean = 0.5393, Acc_std = 0.1043, F1_mean = 0.3987, F1_std = 0.0456, tstNum_mean = 48603.0000, tstNum_std = 0.0000      
2026-08-12 16:13:19.755198: Epoch 100/100, cora Test: Acc_mean = 0.5716, Acc_std = 0.0312, F1_mean = 0.5128, F1_std = 0.0266, tstNum_mean = 3414.0000, tstNum_std = 0.0000      
2026-08-12 16:13:21.540791: Epoch 100/100, home Test: Acc_mean = 0.6178, Acc_std = 0.0504, F1_mean = 0.4088, F1_std = 0.0896, tstNum_mean = 1958.0000, tstNum_std = 0.0000      
2026-08-12 16:13:23.873471: Epoch 100/100, pubmed Test: Acc_mean = 0.7756, Acc_std = 0.0177, F1_mean = 0.7573, F1_std = 0.0211, tstNum_mean = 3944.0000, tstNum_std = 0.0000      
2026-08-12 16:16:14.815788: Epoch 100/100, tech Test: Acc_mea

In [18]:
model_torch = exp.model

Teste das mascaras

In [24]:
def compare_masks(model_mine, model_torch):

    results = []

    modules_torch = dict(model_torch.named_modules())

    for name, param in model_mine.named_parameters():

        if "trainable_nn" not in name or "linear.weight" not in name:
            continue

        module_name = name.replace(".weight", "")

        mask_mine = param.detach() != 0

        mask_torch = modules_torch[
            module_name
        ].weight_mask.detach().bool()

        different = (
            mask_mine.cpu() != mask_torch.cpu()
        ).sum().item()

        total = mask_mine.numel()

        results.append({
            "Layer": name,
            "Different": different,
            "Different (%)": 100 * different / total,
            "Same": different == 0
        })

    return pd.DataFrame(results)

In [25]:
df_masks = compare_masks(
    pruner_mine.model,
    model_torch
)

df_masks

,Layer,Different,Different (%),Same
0,experts.0.trainable_nn.dense_layers.0.linear.w...,0,0.0,True
1,experts.0.trainable_nn.dense_layers.1.linear.w...,0,0.0,True
2,experts.0.trainable_nn.dense_layers.2.linear.w...,0,0.0,True
3,experts.0.trainable_nn.dense_layers.3.linear.w...,0,0.0,True
4,experts.0.trainable_nn.dense_layers.4.linear.w...,0,0.0,True
...,...,...,...,...
59,experts.7.trainable_nn.dense_layers.3.linear.w...,0,0.0,True
60,experts.7.trainable_nn.dense_layers.4.linear.w...,0,0.0,True
61,experts.7.trainable_nn.dense_layers.5.linear.w...,0,0.0,True
62,experts.7.trainable_nn.dense_layers.6.linear.w...,0,0.0,True


In [26]:
print(
    "Camadas idênticas:",
    df_masks["Same"].sum(),
    "/",
    len(df_masks)
)

print(
    "Diferença média:",
    df_masks["Different (%)"].mean()
)

print(
    "Diferença máxima:",
    df_masks["Different (%)"].max()
)

Camadas idênticas: 64 / 64
Diferença média: 0.0
Diferença máxima: 0.0


In [27]:
mask_mine = []
mask_torch = []

for name, param in pruner_mine.model.named_parameters():

    if "trainable_nn" not in name or "linear.weight" not in name:
        continue

    module_name = name.replace(".weight", "")

    mask_mine.append(
        (param.detach() != 0).flatten().cpu()
    )

    mask_torch.append(
        dict(model_torch.named_modules())[
            module_name
        ].weight_mask.detach().bool().flatten().cpu()
    )

mask_mine = t.cat(mask_mine)
mask_torch = t.cat(mask_torch)

print("Nossa sparsidade:",
      100 * (~mask_mine).float().mean().item())

print("PyTorch sparsidade:",
      100 * (~mask_torch).float().mean().item())

print("Máscaras exatamente iguais:",
      t.equal(mask_mine, mask_torch))

Nossa sparsidade: 89.99999761581421
PyTorch sparsidade: 89.99999761581421
Máscaras exatamente iguais: True


In [28]:
def compare_weights(model_mine, model_torch):

    results = []

    modules_torch = dict(model_torch.named_modules())

    for name, param in model_mine.named_parameters():

        if "trainable_nn" not in name or "linear.weight" not in name:
            continue

        module_name = name.replace(".weight", "")

        weight_mine = param.detach().cpu()

        weight_torch = modules_torch[
            module_name
        ].weight.detach().cpu()

        diff = (weight_mine - weight_torch).abs()

        results.append({
            "Layer": name,
            "Max Difference": diff.max().item(),
            "Mean Difference": diff.mean().item(),
            "Different Values": (diff > 0).sum().item()
        })

    return pd.DataFrame(results)

In [29]:
df_weights = compare_weights(
    pruner_mine.model,
    model_torch
)

df_weights

,Layer,Max Difference,Mean Difference,Different Values
0,experts.0.trainable_nn.dense_layers.0.linear.w...,0.0,0.0,0
1,experts.0.trainable_nn.dense_layers.1.linear.w...,0.0,0.0,0
2,experts.0.trainable_nn.dense_layers.2.linear.w...,0.0,0.0,0
3,experts.0.trainable_nn.dense_layers.3.linear.w...,0.0,0.0,0
4,experts.0.trainable_nn.dense_layers.4.linear.w...,0.0,0.0,0
...,...,...,...,...
59,experts.7.trainable_nn.dense_layers.3.linear.w...,0.0,0.0,0
60,experts.7.trainable_nn.dense_layers.4.linear.w...,0.0,0.0,0
61,experts.7.trainable_nn.dense_layers.5.linear.w...,0.0,0.0,0
62,experts.7.trainable_nn.dense_layers.6.linear.w...,0.0,0.0,0


In [30]:
print(
    "Maior diferença:",
    df_weights["Max Difference"].max()
)

print(
    "Diferença média:",
    df_weights["Mean Difference"].mean()
)

print(
    "Valores diferentes:",
    df_weights["Different Values"].sum()
)

Maior diferença: 0.0
Diferença média: 0.0
Valores diferentes: 0


Teste de aleatoriedade

In [31]:
exp = Exp(handler)
exp.prepare_model()
exp.load_model("pretrain_link1")

pruner = GlobalMagnitudePruner(exp.model)

exp.model = pruner.prune(0.90)

df1 = exp.evaluate()
df2 = exp.evaluate()

print("Primeira:")
print(df1)

print("\nSegunda:")
print(df2)

['arxiv', 'cora', 'home', 'pubmed', 'tech']
Test group 0 ['arxiv', 'cora', 'home', 'pubmed', 'tech']
Total params: 16.87552
Trainable params: 16.87552
Non-trainable params: 0.0
2026-08-12 16:31:53.766391: Loaded pretrain_link1
Global Pruning: 90.0%
Threshold: 0.04173007
2026-08-12 16:32:39.977867: Epoch 100/100, arxiv Test: Acc_mean = 0.5504, Acc_std = 0.0432, F1_mean = 0.3621, F1_std = 0.0608, tstNum_mean = 48603.0000, tstNum_std = 0.0000      
2026-08-12 16:32:45.657849: Epoch 100/100, cora Test: Acc_mean = 0.5660, Acc_std = 0.0065, F1_mean = 0.5112, F1_std = 0.0130, tstNum_mean = 3414.0000, tstNum_std = 0.0000      
2026-08-12 16:32:51.723546: Epoch 100/100, home Test: Acc_mean = 0.5948, Acc_std = 0.0655, F1_mean = 0.3685, F1_std = 0.0522, tstNum_mean = 1958.0000, tstNum_std = 0.0000      
2026-08-12 16:32:54.179099: Epoch 100/100, pubmed Test: Acc_mean = 0.7647, Acc_std = 0.0187, F1_mean = 0.7480, F1_std = 0.0200, tstNum_mean = 3944.0000, tstNum_std = 0.0000      
2026-08-12 16:36:

Teste de desempenho do pruning para diferentes espacidades

In [ ]:
import pandas as pd
import torch as t

pruning_levels = [
    0.0, 0.1, 0.2, 0.3, 0.4,
    0.5, 0.6, 0.7, 0.8, 0.9
]

n_repeats = 10
checkpoint = "pretrain_link1"

all_results = []

for sparsity in pruning_levels:

    print("\n" + "=" * 60)
    print(f"Sparsity: {sparsity * 100:.0f}%")
    print("=" * 60)

    for repeat in range(n_repeats):

        print(f"\nRepetição {repeat + 1}/{n_repeats}")

        exp = Exp(handler)

        exp.prepare_model()
        exp.load_model(checkpoint)

        # -----------------------------
        # Pruning
        # -----------------------------

        if sparsity > 0:

            pruner = GlobalMagnitudePruner(exp.model)

            exp.model = pruner.prune(sparsity)

        # -----------------------------
        # Avaliação
        # -----------------------------

        df = exp.evaluate(
            repeat_times=1
        ).copy()

        df["Checkpoint"] = checkpoint
        df["Sparsity"] = sparsity * 100
        df["Repeat"] = repeat + 1

        all_results.append(df)

        del exp

        if sparsity > 0:
            del pruner

        t.cuda.empty_cache()


df_all1 = pd.concat(
    all_results,
    ignore_index=True
)

df_all1


Sparsity: 0%

Repetição 1/10
['arxiv', 'cora', 'home', 'pubmed', 'tech']
Test group 0 ['arxiv', 'cora', 'home', 'pubmed', 'tech']
Total params: 16.87552
Trainable params: 16.87552
Non-trainable params: 0.0
2026-08-12 21:47:17.797401: Loaded pretrain_link1
2026-08-12 21:47:22.079028: Epoch 100/100, arxiv Test: Acc_mean = 0.4677, Acc_std = 0.0000, F1_mean = 0.2925, F1_std = 0.0000, tstNum_mean = 48603.0000, tstNum_std = 0.0000      
2026-08-12 21:47:22.648702: Epoch 100/100, cora Test: Acc_mean = 0.6186, Acc_std = 0.0000, F1_mean = 0.5663, F1_std = 0.0000, tstNum_mean = 3414.0000, tstNum_std = 0.0000      
2026-08-12 21:47:22.955922: Epoch 100/100, home Test: Acc_mean = 0.6578, Acc_std = 0.0000, F1_mean = 0.3977, F1_std = 0.0000, tstNum_mean = 1958.0000, tstNum_std = 0.0000      
2026-08-12 21:47:23.377076: Epoch 100/100, pubmed Test: Acc_mean = 0.7561, Acc_std = 0.0000, F1_mean = 0.7313, F1_std = 0.0000, tstNum_mean = 3944.0000, tstNum_std = 0.0000      
2026-08-12 21:48:09.571580: Epo

,Dataset,Acc_mean,Acc_std,F1_mean,F1_std,tstNum_mean,tstNum_std,Checkpoint,Sparsity,Repeat
0,arxiv,0.467687,0.0,0.292497,0.0,48603.0,0.0,pretrain_link1,0.0,1
1,cora,0.618629,0.0,0.566343,0.0,3414.0,0.0,pretrain_link1,0.0,1
2,home,0.657814,0.0,0.397723,0.0,1958.0,0.0,pretrain_link1,0.0,1
3,pubmed,0.756085,0.0,0.731260,0.0,3944.0,0.0,pretrain_link1,0.0,1
4,tech,0.928421,0.0,0.893005,0.0,9486.0,0.0,pretrain_link1,0.0,1
...,...,...,...,...,...,...,...,...,...,...
595,cora,0.561219,0.0,0.501746,0.0,3414.0,0.0,pretrain_link1,90.0,10
596,home,0.548008,0.0,0.427302,0.0,1958.0,0.0,pretrain_link1,90.0,10
597,pubmed,0.682809,0.0,0.675774,0.0,3944.0,0.0,pretrain_link1,90.0,10
598,tech,0.925364,0.0,0.750744,0.0,9486.0,0.0,pretrain_link1,90.0,10


In [ ]:
df_overall1 = df_all1[
    df_all1["Dataset"] == "OVERALL"
].copy()

summary1 = (
    df_overall1
    .groupby(["Checkpoint", "Sparsity"])
    .agg(
        Acc_mean=("Acc_mean", "mean"),
        Acc_std=("Acc_mean", "std"),
        F1_mean=("F1_mean", "mean"),
        F1_std=("F1_mean", "std")
    )
    .reset_index()
)

summary1

In [ ]:
df_all1.to_csv(
    "pruning_raw_pretrain_link2.csv",
    index=False
)

summary1.to_csv(
    "pruning_summary_pretrain_link2.csv",
    index=False
)

In [ ]:
import pandas as pd
import torch as t

pruning_levels = [
    0.0, 0.1, 0.2, 0.3, 0.4,
    0.5, 0.6, 0.7, 0.8, 0.9
]

n_repeats = 10
checkpoint = "pretrain_link2"

all_results = []

for sparsity in pruning_levels:

    print("\n" + "=" * 60)
    print(f"Sparsity: {sparsity * 100:.0f}%")
    print("=" * 60)

    for repeat in range(n_repeats):

        print(f"\nRepetição {repeat + 1}/{n_repeats}")

        exp = Exp(handler)

        exp.prepare_model()
        exp.load_model(checkpoint)

        # -----------------------------
        # Pruning
        # -----------------------------

        if sparsity > 0:

            pruner = GlobalMagnitudePruner(exp.model)

            exp.model = pruner.prune(sparsity)

        # -----------------------------
        # Avaliação
        # -----------------------------

        df = exp.evaluate(
            repeat_times=1
        ).copy()

        df["Checkpoint"] = checkpoint
        df["Sparsity"] = sparsity * 100
        df["Repeat"] = repeat + 1

        all_results.append(df)

        del exp

        if sparsity > 0:
            del pruner

        t.cuda.empty_cache()


df_all2 = pd.concat(
    all_results,
    ignore_index=True
)

df_all2


Sparsity: 0%

Repetição 1/10
['arxiv', 'cora', 'home', 'pubmed', 'tech']
Test group 0 ['arxiv', 'cora', 'home', 'pubmed', 'tech']
Total params: 16.87552
Trainable params: 16.87552
Non-trainable params: 0.0
2026-08-12 23:29:14.406598: Loaded pretrain_link2
2026-08-12 23:29:18.574753: Epoch 100/100, arxiv Test: Acc_mean = 0.6407, Acc_std = 0.0000, F1_mean = 0.3997, F1_std = 0.0000, tstNum_mean = 48603.0000, tstNum_std = 0.0000      
2026-08-12 23:29:19.152071: Epoch 100/100, cora Test: Acc_mean = 0.6248, Acc_std = 0.0000, F1_mean = 0.5646, F1_std = 0.0000, tstNum_mean = 3414.0000, tstNum_std = 0.0000      
2026-08-12 23:29:19.457066: Epoch 100/100, home Test: Acc_mean = 0.6726, Acc_std = 0.0000, F1_mean = 0.4486, F1_std = 0.0000, tstNum_mean = 1958.0000, tstNum_std = 0.0000      
2026-08-12 23:29:19.875019: Epoch 100/100, pubmed Test: Acc_mean = 0.5948, Acc_std = 0.0000, F1_mean = 0.5473, F1_std = 0.0000, tstNum_mean = 3944.0000, tstNum_std = 0.0000      
2026-08-12 23:29:50.799790: Epo

,Dataset,Acc_mean,Acc_std,F1_mean,F1_std,tstNum_mean,tstNum_std,Checkpoint,Sparsity,Repeat
0,arxiv,0.640681,0.0,0.399745,0.0,48603.0,0.0,pretrain_link2,0.0,1
1,cora,0.624780,0.0,0.564576,0.0,3414.0,0.0,pretrain_link2,0.0,1
2,home,0.672625,0.0,0.448605,0.0,1958.0,0.0,pretrain_link2,0.0,1
3,pubmed,0.594828,0.0,0.547260,0.0,3944.0,0.0,pretrain_link2,0.0,1
4,tech,0.627873,0.0,0.366837,0.0,9486.0,0.0,pretrain_link2,0.0,1
...,...,...,...,...,...,...,...,...,...,...
595,cora,0.617458,0.0,0.556859,0.0,3414.0,0.0,pretrain_link2,90.0,10
596,home,0.734934,0.0,0.514504,0.0,1958.0,0.0,pretrain_link2,90.0,10
597,pubmed,0.649087,0.0,0.591828,0.0,3944.0,0.0,pretrain_link2,90.0,10
598,tech,0.874657,0.0,0.833903,0.0,9486.0,0.0,pretrain_link2,90.0,10


In [11]:
df_overall2 = df_all[
    df_all["Dataset"] == "OVERALL"
].copy()

summary2 = (
    df_overall2
    .groupby(["Checkpoint", "Sparsity"])
    .agg(
        Acc_mean=("Acc_mean", "mean"),
        Acc_std=("Acc_mean", "std"),
        F1_mean=("F1_mean", "mean"),
        F1_std=("F1_mean", "std")
    )
    .reset_index()
)

summary2

,Checkpoint,Sparsity,Acc_mean,Acc_std,F1_mean,F1_std
0,pretrain_link2,0.0,0.648964,0.021732,0.429332,0.041083
1,pretrain_link2,10.0,0.651544,0.017688,0.442988,0.029925
2,pretrain_link2,20.0,0.651173,0.018018,0.437963,0.035522
3,pretrain_link2,30.0,0.656104,0.013865,0.452426,0.025211
4,pretrain_link2,40.0,0.656696,0.015484,0.450927,0.026588
5,pretrain_link2,50.0,0.658742,0.016768,0.464640,0.030472
6,pretrain_link2,60.0,0.649477,0.014740,0.475853,0.021831
7,pretrain_link2,70.0,0.650370,0.015239,0.481704,0.029183
8,pretrain_link2,80.0,0.659570,0.012983,0.495667,0.021567
9,pretrain_link2,90.0,0.663943,0.012252,0.504859,0.013507


In [ ]:
df_all2.to_csv(
    "pruning_raw_pretrain_link2.csv",
    index=False
)

summary2.to_csv(
    "pruning_summary_pretrain_link2.csv",
    index=False
)

Para reorganizar o codigoa baixo

In [13]:
report = pruner.sparsity_report()

report

{'Threshold': 0.005316588096320629,
 'Requested Sparsity (%)': 30.0,
 'Total': 16777216,
 'Zero': 5033166,
 'Remaining': 11744050,
 'Sparsity (%)': 30.000007152557373}

In [11]:
checkpoint="pretrain_link1"

exp = Exp(handler)

exp.prepare_model()

exp.load_model(checkpoint)

pruner = GlobalMagnitudePruner(exp.model)

pruner.prune(0.30)

exp.model = pruner.get_model()

df = exp.evaluate()

print(df)

['arxiv', 'cora', 'home', 'pubmed', 'tech']
Test group 0 ['arxiv', 'cora', 'home', 'pubmed', 'tech']
Total params: 16.87552
Trainable params: 16.87552
Non-trainable params: 0.0
2026-08-10 17:15:21.693231: Loaded pretrain_link1
Global Magnitude Pruning: 30.0%
Threshold = 0.00531659
2026-08-10 17:15:43.967969: Epoch 100/100, arxiv Test: Acc_mean = 0.4937, Acc_std = 0.0226, F1_mean = 0.3091, F1_std = 0.0098, tstNum_mean = 48603.0000, tstNum_std = 0.0000      
2026-08-10 17:15:46.584746: Epoch 100/100, cora Test: Acc_mean = 0.6113, Acc_std = 0.0193, F1_mean = 0.5522, F1_std = 0.0179, tstNum_mean = 3414.0000, tstNum_std = 0.0000      
2026-08-10 17:15:48.139989: Epoch 100/100, home Test: Acc_mean = 0.6380, Acc_std = 0.0707, F1_mean = 0.3920, F1_std = 0.0947, tstNum_mean = 1958.0000, tstNum_std = 0.0000      
2026-08-10 17:15:50.255801: Epoch 100/100, pubmed Test: Acc_mean = 0.7130, Acc_std = 0.0498, F1_mean = 0.6861, F1_std = 0.0522, tstNum_mean = 3944.0000, tstNum_std = 0.0000      
2026-0

In [12]:
for checkpoint in ["pretrain_link1", "pretrain_link2"]:
    exp = Exp(handler)

    exp.prepare_model()

    exp.load_model(checkpoint)

    df = exp.evaluate()

    print(df)

['arxiv', 'cora', 'home', 'pubmed', 'tech']
Test group 0 ['arxiv', 'cora', 'home', 'pubmed', 'tech']
Total params: 16.87552
Trainable params: 16.87552
Non-trainable params: 0.0
2026-08-07 17:46:31.639752: Loaded pretrain_link1
2026-08-07 17:46:52.152254: Epoch 100/100, arxiv Test: Acc_mean = 0.5024, Acc_std = 0.0125, F1_mean = 0.3010, F1_std = 0.0157, tstNum_mean = 48603.0000, tstNum_std = 0.0000      
2026-08-07 17:46:54.737893: Epoch 100/100, cora Test: Acc_mean = 0.6062, Acc_std = 0.0211, F1_mean = 0.5516, F1_std = 0.0208, tstNum_mean = 3414.0000, tstNum_std = 0.0000      
2026-08-07 17:46:56.286861: Epoch 100/100, home Test: Acc_mean = 0.6520, Acc_std = 0.0656, F1_mean = 0.4375, F1_std = 0.1014, tstNum_mean = 1958.0000, tstNum_std = 0.0000      
2026-08-07 17:46:58.387732: Epoch 100/100, pubmed Test: Acc_mean = 0.7145, Acc_std = 0.0488, F1_mean = 0.6738, F1_std = 0.0542, tstNum_mean = 3944.0000, tstNum_std = 0.0000      
2026-08-07 17:49:33.303974: Epoch 100/100, tech Test: Acc_mea

In [ ]:
exp = Exp(handler)
    
exp.prepare_model()
    
exp.load_model(checkpoint)

pruner = GlobalMagnitudePruner(exp.model)

pruner.prune(0.10)

df = exp.evaluate()

print(df)

In [18]:
for checkpoint in ["pretrain_link1", "pretrain_link2"]:

    exp = Exp(handler)
    
    exp.prepare_model()
    
    exp.load_model(checkpoint)

    pruner = GlobalMagnitudePruner(exp.model)

    pruner.prune(0.10)

    df = exp.evaluate()

    print(checkpoint)

    print(df)

['arxiv', 'cora', 'home', 'pubmed', 'tech']
Test group 0 ['arxiv', 'cora', 'home', 'pubmed', 'tech']
Total params: 16.87552
Trainable params: 16.87552
Non-trainable params: 0.0
2026-08-07 19:02:50.730356: Loaded pretrain_link1
Threshold = 0.000820
2026-08-07 19:03:11.009694: Epoch 100/100, arxiv Test: Acc_mean = 0.5064, Acc_std = 0.0075, F1_mean = 0.3056, F1_std = 0.0087, tstNum_mean = 48603.0000, tstNum_std = 0.0000      
2026-08-07 19:03:13.544945: Epoch 100/100, cora Test: Acc_mean = 0.6040, Acc_std = 0.0213, F1_mean = 0.5477, F1_std = 0.0166, tstNum_mean = 3414.0000, tstNum_std = 0.0000      
2026-08-07 19:03:15.045403: Epoch 100/100, home Test: Acc_mean = 0.6524, Acc_std = 0.0437, F1_mean = 0.4926, F1_std = 0.0739, tstNum_mean = 1958.0000, tstNum_std = 0.0000      
2026-08-07 19:03:17.130510: Epoch 100/100, pubmed Test: Acc_mean = 0.6745, Acc_std = 0.0419, F1_mean = 0.6356, F1_std = 0.0377, tstNum_mean = 3944.0000, tstNum_std = 0.0000      
2026-08-07 19:05:31.113575: Epoch 100/10

In [19]:
for checkpoint in ["pretrain_link1", "pretrain_link2"]:

    exp = Exp(handler)
    
    exp.prepare_model()
    
    exp.load_model(checkpoint)

    pruner = GlobalMagnitudePruner(exp.model)

    pruner.prune(0.20)

    df = exp.evaluate()

    print(checkpoint)

    print(df)

['arxiv', 'cora', 'home', 'pubmed', 'tech']
Test group 0 ['arxiv', 'cora', 'home', 'pubmed', 'tech']
Total params: 16.87552
Trainable params: 16.87552
Non-trainable params: 0.0
2026-08-07 19:08:44.650856: Loaded pretrain_link1
Threshold = 0.002704
2026-08-07 19:09:04.846822: Epoch 100/100, arxiv Test: Acc_mean = 0.4982, Acc_std = 0.0090, F1_mean = 0.3053, F1_std = 0.0109, tstNum_mean = 48603.0000, tstNum_std = 0.0000      
2026-08-07 19:09:07.391998: Epoch 100/100, cora Test: Acc_mean = 0.5954, Acc_std = 0.0248, F1_mean = 0.5396, F1_std = 0.0222, tstNum_mean = 3414.0000, tstNum_std = 0.0000      
2026-08-07 19:09:08.895218: Epoch 100/100, home Test: Acc_mean = 0.6284, Acc_std = 0.0714, F1_mean = 0.4160, F1_std = 0.1203, tstNum_mean = 1958.0000, tstNum_std = 0.0000      
2026-08-07 19:09:10.969125: Epoch 100/100, pubmed Test: Acc_mean = 0.7204, Acc_std = 0.0576, F1_mean = 0.6977, F1_std = 0.0553, tstNum_mean = 3944.0000, tstNum_std = 0.0000      
2026-08-07 19:11:22.543953: Epoch 100/10

In [8]:
for checkpoint in ["pretrain_link1", "pretrain_link2"]:

    exp = Exp(handler)

    exp.prepare_model()

    exp.load_model("pretrain_link1")

    pruner = GlobalMagnitudePruner(exp.model)

    pruner.prune(0.30)

    exp.model = pruner.get_model()

    df = exp.evaluate()

    print(df)
    print(30*"==")

['arxiv', 'cora', 'home', 'pubmed', 'tech']
Test group 0 ['arxiv', 'cora', 'home', 'pubmed', 'tech']
Total params: 16.87552
Trainable params: 16.87552
Non-trainable params: 0.0
2026-08-07 19:39:58.539014: Loaded pretrain_link1
Global Magnitude Pruning: 30.0%
Threshold = 0.00531659
2026-08-07 19:40:19.090462: Epoch 100/100, arxiv Test: Acc_mean = 0.4950, Acc_std = 0.0123, F1_mean = 0.2956, F1_std = 0.0073, tstNum_mean = 48603.0000, tstNum_std = 0.0000      
2026-08-07 19:40:21.764268: Epoch 100/100, cora Test: Acc_mean = 0.6067, Acc_std = 0.0166, F1_mean = 0.5517, F1_std = 0.0179, tstNum_mean = 3414.0000, tstNum_std = 0.0000      
2026-08-07 19:40:23.256014: Epoch 100/100, home Test: Acc_mean = 0.6795, Acc_std = 0.0419, F1_mean = 0.4548, F1_std = 0.0647, tstNum_mean = 1958.0000, tstNum_std = 0.0000      
2026-08-07 19:40:25.351067: Epoch 100/100, pubmed Test: Acc_mean = 0.6591, Acc_std = 0.0507, F1_mean = 0.6186, F1_std = 0.0671, tstNum_mean = 3944.0000, tstNum_std = 0.0000      
2026-0